# FanDuel OSB cross-state research study

**Audience:** data scientist / CFA analyst.  
**Question:** Is FanDuel’s handle-share weakness broad, and does recent improvement survive sample, calendar, and comparison-period checks?

| Layer | States / scope |
|---|---|
| **Baseline exploratory** | KS, MI, WY, MA (prior technical checks) |
| **Formal approval** | **MA only** (notebook 91) |
| **Expanded / separate** | OH (transfer), IN (derived denom), NY (weekly), NJ (revenue-only) |

Online only. Not nationally representative. No customer-migration or valuation claims.


## 0. Setup (read-only)


In [ ]:
from __future__ import annotations

import hashlib
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
import sys

sys.path.insert(0, str(ROOT / "src"))
from variant_gaming.consolidate import PRINTED_OFFICIAL_PROVENANCE
from variant_gaming.storage import connect_readonly

STAGING_DB = ROOT / "data/staging/gaming_nationwide.sqlite"
ORIGINAL_DB = ROOT / "data/gaming.sqlite"
EXPECTED_STAGING = "023ca5e8e4c16ff0981a2783dabcedf9399939e0241701b0394bc0277eff6ce9"
APPROVED_PIPELINE = "2e752d47e810b7aaf583d534ab5a411aa7fa1c1f"
BIND = (
    "src/variant_gaming/consolidate.py",
    "src/variant_gaming/storage.py",
    "src/variant_gaming/states/massachusetts.py",
    "config/state_metric_notes.csv",
    "config/state_gaming_source_inventory.csv",
)
VERTICAL, CHANNEL, FREQ = "online_sports_betting", "online", "monthly"
RECON_TOL = 5.0
BASELINE = ["KS", "MI", "WY", "MA"]
NATIVE = {
    "MA": ["FanDuel"],
    "WY": ["FanDuel"],
    "MI": ["FanDuel (MotorCity Casino)"],
    "KS": ["Kansas Star FanDuel"],
    "IN": ["BC - in.sportsbook.FanDuel.com"],
    "OH": ["BELTERRA PARK - FANDUEL", "HOLLYWOOD MAHONING VALLEY - FANDUEL"],
}
C_BLUE, C_ORANGE, C_GRAY, C_BLACK = "#0072B2", "#E69F00", "#4D4D4D", "#000000"
WORLD_CUP = ("2026-06-11", "2026-07-19")  # FIFA retained-source annotation only
WC_URL = "https://gpcustomersupportfwc2026.tickets.fifa.com/hc/en-gb/articles/28783291386653-1-When-and-where-is-the-FIFA-World-Cup-2026-being-held"


def sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def git_out(*a: str) -> str:
    return subprocess.check_output(["git", *a], cwd=ROOT, text=True).strip()


def is_derived(state: str, status: str) -> bool:
    printed = status in PRINTED_OFFICIAL_PROVENANCE
    return status == "derived_from_operator_sum" or (state in {"IL", "IN"} and not printed)


h0, s0 = sha256(ORIGINAL_DB), sha256(STAGING_DB)
if s0 != EXPECTED_STAGING:
    raise SystemExit("FAIL CLOSED: analyst-approval binding mismatch: staging snapshot changed; review required.")
for rel in BIND:
    if git_out("hash-object", rel) != git_out("rev-parse", f"{APPROVED_PIPELINE}:{rel}"):
        raise SystemExit(f"FAIL CLOSED: analyst-approval binding mismatch: {rel} changed; review required.")
print("staging", s0)
print("original", h0)
print("MA formal approval unchanged; other work exploratory.")
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 120)


## 1. Load OSB observations


In [ ]:
conn = connect_readonly(STAGING_DB)
raw = pd.read_sql_query(
    """
    SELECT state_code, channel, frequency, row_type, operator, period_start, period_end,
           handle, gross_revenue, report_status, source_sha256, source_file, source_url,
           reported_revenue_name
    FROM gaming_results WHERE vertical=?
    """,
    conn,
    params=(VERTICAL,),
)
conn.close()
raw["period_start"] = pd.to_datetime(raw["period_start"])
raw["period_end"] = pd.to_datetime(raw["period_end"])
monthly = raw[raw["channel"].eq(CHANNEL) & raw["frequency"].eq(FREQ)].copy()


def panel(state: str, names: list[str], allow_derived: bool = False) -> pd.DataFrame:
    g = monthly[monthly["state_code"].eq(state)]
    ops = g[g["row_type"].eq("operator")]
    off = g[g["row_type"].eq("official_statewide_total")]
    fd = ops[ops["operator"].isin(names)]
    rows = []
    for ts, fdr in fd.groupby("period_start"):
        if fdr["handle"].isna().any():
            continue
        if state != "OH" and len(fdr) != 1:
            continue
        offr = off[off["period_start"].eq(ts)]
        if len(offr) != 1 or pd.isna(offr["handle"].iloc[0]):
            continue
        status = str(offr["report_status"].iloc[0])
        derived = is_derived(state, status)
        if derived and not allow_derived:
            continue
        if (not derived) and status not in PRINTED_OFFICIAL_PROVENANCE and not status:
            pass
        mkt = float(offr["handle"].iloc[0])
        if mkt <= 0:
            continue
        opr = ops[ops["period_start"].eq(ts)]
        if opr["handle"].isna().any() or opr.operator.duplicated().any():
            continue
        op_sum = float(opr["handle"].sum())
        if abs(op_sum - mkt) > RECON_TOL:
            continue
        rows.append(
            {
                "state": state,
                "month": ts,
                "identity": " + ".join(sorted(fdr["operator"].unique())),
                "fd_handle": float(fdr["handle"].sum(min_count=1)),
                "mkt_handle": mkt,
                "share": float(fdr["handle"].sum(min_count=1)) / mkt,
                "report_status": status,
                "derived": derived,
                "denom": (
                    "derived_operator_sum"
                    if derived
                    else (
                        f"printed:{status}"
                        if status in PRINTED_OFFICIAL_PROVENANCE
                        else f"retained_official:{status}"
                    )
                ),
            }
        )
    return pd.DataFrame(rows, columns=["state", "month", "identity", "fd_handle", "mkt_handle", "share", "report_status", "derived", "denom"]).sort_values("month")


panels = {st: panel(st, NATIVE[st], allow_derived=False) for st in BASELINE}
panels["OH"] = panel("OH", NATIVE["OH"], allow_derived=False)
panels["IN"] = panel("IN", NATIVE["IN"], allow_derived=True)

for st, p in panels.items():
    print(f"{st}: {len(p)} months {p.month.min().date() if len(p) else None} → {p.month.max().date() if len(p) else None} derived={p.derived.any() if len(p) else None}")


## 2. Inclusion recommendations (expanded candidates)

| Candidate | Recommendation | Reason |
|---|---|---|
| **OH** | **Expanded exploratory** (sum Belterra+Hollywood) | Feb 2026 handle moves Belterra→0 / Hollywood starts; same-brand continuity is the working hypothesis, not proven corporate identity |
| **IN** | **Separate sensitivity only** | `report_status=derived_from_operator_sum`; stable 11 operators/month but ESPNBet→theScore name change; self-recon ≠ independent completeness proof |
| **NY** | **Separate weekly robustness** | Weekly cash-basis; do not allocate into calendar months |
| **NJ** | **Separate OSB revenue panel** | Handle missing; revenue labels shift in 2026 |


In [ ]:
# OH transfer evidence (handle path)
oh = monthly[
    monthly["state_code"].eq("OH")
    & monthly["row_type"].eq("operator")
    & monthly["operator"].isin(NATIVE["OH"])
    & monthly["period_start"].between("2025-11-01", "2026-07-01")
][["period_start", "operator", "handle"]].sort_values(["period_start", "operator"])
display(Markdown("### OH FanDuel licensee handles (transfer window)"))
display(oh)

# IN roster July comparison
def in_ops(ts):
    return set(
        monthly[
            monthly["state_code"].eq("IN")
            & monthly["row_type"].eq("operator")
            & monthly["period_start"].eq(ts)
        ]["operator"]
    )


print("IN Jul2025 ops", len(in_ops(pd.Timestamp("2025-07-01"))))
print("IN Jul2026 ops", len(in_ops(pd.Timestamp("2026-07-01"))))
print("only 2025", sorted(in_ops(pd.Timestamp("2025-07-01")) - in_ops(pd.Timestamp("2026-07-01"))))
print("only 2026", sorted(in_ops(pd.Timestamp("2026-07-01")) - in_ops(pd.Timestamp("2025-07-01"))))


## 3. Coverage, sources, and fixed comparison
Handle growth describes wager volume; handle share describes FanDuel's share of that volume. Neither is revenue growth or hold. Notebook 91 separates those effects for MA using Accrual Win; notebook 93 uses MI casino Gross Receipts without a hold claim. Other states remain exploratory; OH's combined licensees remain a disclosed hypothesis.

The table shows rejected and absent months. Missing values, duplicate identities/source versions, and failed reconciliation need source review. A smaller surviving sample is not permission to substitute a different window.

Required baseline states must have every matched month before calculation. Incomplete OH expansion is skipped; IN remains separately conditional.


In [ ]:
PRIOR = [p.to_timestamp() for p in pd.period_range("2025-01", "2025-07", freq="M")]
CURRENT = [p.to_timestamp() for p in pd.period_range("2026-01", "2026-07", freq="M")]
JAN_MAY_P = [p.to_timestamp() for p in pd.period_range("2025-01", "2025-05", freq="M")]
JAN_MAY_C = [p.to_timestamp() for p in pd.period_range("2026-01", "2026-05", freq="M")]
JUN_JUL_P = [p.to_timestamp() for p in pd.period_range("2025-06", "2025-07", freq="M")]
JUN_JUL_C = [p.to_timestamp() for p in pd.period_range("2026-06", "2026-07", freq="M")]


audit_rows = []
for state, panel_rows in panels.items():
    for month in PRIOR + CURRENT:
        source_rows = monthly[monthly.state_code.eq(state) & monthly.period_start.eq(month)]
        usable = month in set(panel_rows.month)
        audit_rows.append({"state": state, "month": month, "eligible": usable,
            "status": "included" if usable else ("no stored rows" if source_rows.empty else "rejected by panel checks"),
            "missing_handle": int(source_rows.handle.isna().sum()),
            "duplicate_keys": int(source_rows.duplicated(["row_type", "operator"], keep=False).sum()),
            "source_file": list(source_rows.source_file.unique()),
            "source_url": list(source_rows.source_url.unique())})
comparison_coverage = pd.DataFrame(audit_rows)
display(comparison_coverage)

missing_required = comparison_coverage[
    comparison_coverage.state.isin(BASELINE) & ~comparison_coverage.eligible]
if not missing_required.empty:
    missing_periods = "; ".join(
        f"{state}: {', '.join(group.month.dt.strftime('%Y-%m'))}"
        for state, group in missing_required.groupby("state"))
    raise SystemExit("FAIL CLOSED: required baseline months missing or rejected: "
                     + missing_periods + ". Inspect comparison_coverage and retained sources.")


def window_ok(p: pd.DataFrame, months: list) -> bool:
    return set(months).issubset(set(p["month"]))


def pooled(p: pd.DataFrame, months: list) -> tuple[float, float, float]:
    s = p[p["month"].isin(months)]
    fd, mkt = float(s["fd_handle"].sum()), float(s["mkt_handle"].sum())
    return fd, mkt, fd / mkt


def state_summary(states: list[str], cur_m, pri_m, label: str) -> pd.DataFrame:
    rows = []
    for st in states:
        p = panels[st]
        if not (window_ok(p, cur_m) and window_ok(p, pri_m)):
            continue
        fd1, m1, s1 = pooled(p, cur_m)
        fd0, m0, s0 = pooled(p, pri_m)
        rows.append(
            {
                "sample": label,
                "state": st,
                "role": "formal_approval" if st == "MA" else "exploratory",
                "share_prior": s0,
                "share_cur": s1,
                "share_pp": 100 * (s1 - s0),
                "fd_yoy": 100 * (fd1 / fd0 - 1),
                "mkt_yoy": 100 * (m1 / m0 - 1),
            }
        )
    return pd.DataFrame(rows)


def weighted(states, cur_m, pri_m):
    parts = []
    for st in states:
        p = panels[st]
        if not (window_ok(p, cur_m) and window_ok(p, pri_m)):
            return None
        parts.append(p[p["month"].isin(cur_m + pri_m)].assign(_st=st))
    allp = pd.concat(parts)
    cur = allp[allp["month"].isin(cur_m)]
    pri = allp[allp["month"].isin(pri_m)]
    s1 = float(cur["fd_handle"].sum() / cur["mkt_handle"].sum())
    s0 = float(pri["fd_handle"].sum() / pri["mkt_handle"].sum())
    # constant mix
    pri_st = pri.groupby("state").agg(fd=("fd_handle", "sum"), mkt=("mkt_handle", "sum"))
    cur_st = cur.groupby("state").agg(fd=("fd_handle", "sum"), mkt=("mkt_handle", "sum"))
    st = pri_st.join(cur_st, lsuffix="_0", rsuffix="_1")
    st["dpp"] = 100 * (st["fd_1"] / st["mkt_1"] - st["fd_0"] / st["mkt_0"])
    st["w0"] = st["mkt_0"] / st["mkt_0"].sum()
    cm = float((st["w0"] * st["dpp"]).sum())
    pooled_pp = 100 * (s1 - s0)
    return {
        "pooled_share_pp": pooled_pp,
        "constant_mix_pp": cm,
        "mix_effect_pp": pooled_pp - cm,
        "share_0": s0,
        "share_1": s1,
        "n": len(states),
        "n_loss": int((st["dpp"] < -0.05).sum()),
        "n_gain": int((st["dpp"] > 0.05).sum()),
    }


base_jj = state_summary(BASELINE, CURRENT, PRIOR, "baseline")
ex_ks = state_summary([s for s in BASELINE if s != "KS"], CURRENT, PRIOR, "ex_KS")
oh_complete = window_ok(panels["OH"], CURRENT) and window_ok(panels["OH"], PRIOR)
exp_jj = (state_summary(BASELINE + ["OH"], CURRENT, PRIOR, "baseline+OH")
          if oh_complete else base_jj.iloc[:0].copy())
display(Markdown("### Jan–Jul state share changes"))
for df in (base_jj, ex_ks, exp_jj):
    v = df.copy()
    v["share_prior"] = v["share_prior"].map(lambda x: f"{100*x:.2f}%")
    v["share_cur"] = v["share_cur"].map(lambda x: f"{100*x:.2f}%")
    v["share_pp"] = v["share_pp"].map(lambda x: f"{x:+.2f} pp")
    v["fd_yoy"] = v["fd_yoy"].map(lambda x: f"{x:+.2f}%")
    v["mkt_yoy"] = v["mkt_yoy"].map(lambda x: f"{x:+.2f}%")
    display(v)

w_base = weighted(BASELINE, CURRENT, PRIOR)
w_ex = weighted([s for s in BASELINE if s != "KS"], CURRENT, PRIOR)
w_oh = weighted(BASELINE + ["OH"], CURRENT, PRIOR)
print("Baseline weighted Jan–Jul", {k: round(v, 2) if isinstance(v, float) else v for k, v in w_base.items()})
print("ex-KS", {k: round(v, 2) if isinstance(v, float) else v for k, v in w_ex.items()})
if w_oh is None:
    print("Expanded baseline+OH skipped: OH matched coverage incomplete; see comparison_coverage.")
else:
    print("baseline+OH", {k: round(v, 2) if isinstance(v, float) else v for k, v in w_oh.items()})

ma = base_jj[base_jj.state.eq("MA")].iloc[0]
assert abs(ma.share_pp - (-2.04)) < 0.02 and abs(ma.fd_yoy - (-4.97)) < 0.02
print("MA Jan–Jul parity vs notebook 91: OK")


## 4. Timing / seasonality checks

Compare Jan–May vs June–July (matched months), then inspect **monthly** and trailing-3M pooled YoY share changes.
Distinguish **improvement** (less-negative YoY share change) from a **positive** share-change sign.
Annotate FIFA World Cup 2026 dates **11 Jun–19 Jul 2026** as context only — monthly bins do **not** isolate tournament days  
([FIFA customer-support article](https://gpcustomersupportfwc2026.tickets.fifa.com/hc/en-gb/articles/28783291386653-1-When-and-where-is-the-FIFA-World-Cup-2026-being-held)).


In [ ]:
timing_rows = []
for name, cur_m, pri_m in (
    ("Jan-May", JAN_MAY_C, JAN_MAY_P),
    ("Jun-Jul", JUN_JUL_C, JUN_JUL_P),
    ("Jan-Jul", CURRENT, PRIOR),
    ("July", [pd.Timestamp("2026-07-01")], [pd.Timestamp("2025-07-01")]),
):
    w = weighted(BASELINE, cur_m, pri_m)
    wx = weighted([s for s in BASELINE if s != "KS"], cur_m, pri_m)
    timing_rows.append({"window": name, "sample": "baseline", **w})
    timing_rows.append({"window": name, "sample": "ex_KS", **wx})

timing = pd.DataFrame(timing_rows)
display(Markdown("### Pooled / constant-mix by calendar window"))
tv = timing.copy()
for c in ["pooled_share_pp", "constant_mix_pp", "mix_effect_pp", "share_0", "share_1"]:
    if c.startswith("share_"):
        tv[c] = tv[c].map(lambda x: f"{100*x:.2f}%")
    else:
        tv[c] = tv[c].map(lambda x: f"{x:+.2f} pp")
display(tv)
print(
    f"World Cup annotation only: {WORLD_CUP[0]} → {WORLD_CUP[1]} ({WC_URL}). "
    "Jun–Jul monthly comparison is NOT a causal World Cup estimate."
)

# Monthly pooled YoY share change (evidence for timing; not a new chart)
month_rows = []
for cur, pri in zip(CURRENT, PRIOR):
    wb = weighted(BASELINE, [cur], [pri])
    wx = weighted([s for s in BASELINE if s != "KS"], [cur], [pri])
    month_rows.append(
        {
            "month": cur.strftime("%b"),
            "baseline_pp": wb["pooled_share_pp"],
            "ex_KS_pp": wx["pooled_share_pp"],
        }
    )
month_pp = pd.DataFrame(month_rows)
display(Markdown("### Monthly pooled YoY handle-share change (summed dollars)"))
mv = month_pp.copy()
mv["baseline_pp"] = mv["baseline_pp"].map(lambda x: f"{x:+.2f} pp")
mv["ex_KS_pp"] = mv["ex_KS_pp"].map(lambda x: f"{x:+.2f} pp")
display(mv)

t3_pool_rows = []
for end in pd.period_range("2026-03", "2026-07", freq="M"):
    cur_ps = pd.period_range(end - 2, periods=3, freq="M")
    pri_ps = pd.period_range(end - 2 - 12, periods=3, freq="M")
    cur_m = [x.to_timestamp() for x in cur_ps]
    pri_m = [x.to_timestamp() for x in pri_ps]
    if not all(window_ok(panels[s], cur_m) and window_ok(panels[s], pri_m) for s in BASELINE):
        continue
    wb = weighted(BASELINE, cur_m, pri_m)
    wx = weighted([s for s in BASELINE if s != "KS"], cur_m, pri_m)
    t3_pool_rows.append(
        {
            "t3m_end": str(end),
            "baseline_pp": wb["pooled_share_pp"],
            "ex_KS_pp": wx["pooled_share_pp"],
        }
    )
t3_pool = pd.DataFrame(t3_pool_rows)
display(Markdown("### Trailing-3M pooled YoY handle-share change (complete consecutive months)"))
tv3 = t3_pool.copy()
tv3["baseline_pp"] = tv3["baseline_pp"].map(lambda x: f"{x:+.2f} pp")
tv3["ex_KS_pp"] = tv3["ex_KS_pp"].map(lambda x: f"{x:+.2f} pp")
display(tv3)

# Small multiples: monthly share, separate yearly lines
fig, axes = plt.subplots(2, 2, figsize=(9.5, 6.0), sharex=True)
axes = axes.ravel()
months_abbr = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul"]
for ax, st in zip(axes, BASELINE):
    p = panels[st]
    y25 = p[p["month"].isin(PRIOR)].sort_values("month")
    y26 = p[p["month"].isin(CURRENT)].sort_values("month")
    ax.plot(months_abbr, 100 * y25["share"].to_numpy(), marker="o", color=C_GRAY, label="2025")
    ax.plot(months_abbr, 100 * y26["share"].to_numpy(), marker="o", color=C_BLUE, label="2026")
    ax.axvspan(5 - 0.4, 6 + 0.4, color=C_ORANGE, alpha=0.12, label="Jun–Jul (WC overlaps 2026)")
    ax.set_title(f"{st} ({'approved' if st=='MA' else 'exploratory'})")
    ax.set_ylabel("Handle share %")
    ax.legend(fontsize=7, frameon=False)
fig.suptitle("Baseline monthly FanDuel handle share — yearly lines (not connected across years)", y=1.02)
fig.tight_layout()
plt.show()

# Trailing 3-month share change vs prior-year 3-month window
t3_rows = []
for st in BASELINE:
    p = panels[st].set_index("month").sort_index()
    for end in pd.period_range("2025-03", "2026-07", freq="M"):
        cur_ps = pd.period_range(end - 2, periods=3, freq="M")
        pri_ps = pd.period_range(end - 2 - 12, periods=3, freq="M")
        cur_months = [x.to_timestamp() for x in cur_ps]
        pri_months = [x.to_timestamp() for x in pri_ps]
        if not all(m in p.index for m in cur_months + pri_months):
            continue
        fd1 = float(p.loc[cur_months, "fd_handle"].sum())
        m1 = float(p.loc[cur_months, "mkt_handle"].sum())
        fd0 = float(p.loc[pri_months, "fd_handle"].sum())
        m0 = float(p.loc[pri_months, "mkt_handle"].sum())
        t3_rows.append(
            {
                "state": st,
                "end_month": end.to_timestamp(),
                "share_pp": 100 * (fd1 / m1 - fd0 / m0),
            }
        )

t3 = pd.DataFrame(t3_rows)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for st, color in zip(BASELINE, [C_BLUE, C_ORANGE, C_GRAY, "#009E73"]):
    s = t3[t3.state.eq(st)].sort_values("end_month")
    ax.plot(s["end_month"], s["share_pp"], marker="o", label=st, color=color, linewidth=1.6)
ax.axhline(0, color=C_BLACK, linewidth=0.8)
ax.axvspan(pd.Timestamp("2026-06-01"), pd.Timestamp("2026-07-01"), color=C_ORANGE, alpha=0.15)
ax.set_title("Trailing 3-month handle-share change vs prior-year 3-month window (summed dollars)")
ax.set_ylabel("Share change (pp)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
print("Rolling T3M smooths noise; it is not seasonal adjustment.")


## 5. Sports-mix identification limit

No sport-level or operator-by-sport fields exist in `gaming_results`.  
**Cannot** attribute FanDuel aggregate share moves to changing sport weights. That remains an open identification limit.


## 6. Separate checks — NY weekly, NJ revenue, IN sensitivity

NY requires every Monday–Sunday key wholly inside each window, one observation per key in both series, and complete handle values. Coverage counts observed weeks × seven days, excluding boundary straddlers; the diagnostics appear before comparison.

NJ requires every requested month, exactly one FanDuel row per month, and no duplicate operator versions. Native operator names and retained file references remain visible. The retained rows do not establish an independently complete roster, so the denominator is an **unverified observed-operator denominator**, not certified complete market share. Missing FanDuel observations block calculation rather than becoming zero.


In [ ]:
# NY: weeks fully inside Jan–Jul (no month allocation)
ny_fd = raw[
    raw["state_code"].eq("NY")
    & raw["frequency"].eq("weekly")
    & raw["operator"].eq("FanDuel")
    & raw["row_type"].eq("operator")
].copy()
ny_mkt = raw[
    raw["state_code"].eq("NY")
    & raw["frequency"].eq("weekly")
    & raw["row_type"].eq("official_statewide_total")
].copy()


def ny_window(year: int):
    """Require a consecutive, matched NY reporting grid fully inside Jan 1–Jul 31."""
    lo, hi = pd.Timestamp(f"{year}-01-01"), pd.Timestamp(f"{year}-07-31")
    fd_all = ny_fd[(ny_fd.period_start <= hi) & (ny_fd.period_end >= lo)].copy()
    fd = ny_fd[(ny_fd.period_start >= lo) & (ny_fd.period_end <= hi)].copy()
    mk = ny_mkt[(ny_mkt.period_start >= lo) & (ny_mkt.period_end <= hi)].copy()
    # Retained NY workbooks use Monday–Sunday weeks. Check every expected key.
    keys = pd.date_range(lo, hi - pd.Timedelta(days=6), freq="W-MON")
    expected = set(zip(keys, keys + pd.Timedelta(days=6)))
    checks = []
    for name, frame in (("FanDuel", fd), ("market", mk)):
        observed = set(zip(frame.period_start, frame.period_end))
        bad_values = frame.handle.isna() | frame.handle.isin([float("inf"), -float("inf")])
        checks.append(dict(series=name, year=year, expected_weeks=len(keys), observed_rows=len(frame),
            missing_keys=sorted(expected - observed), unexpected_keys=sorted(observed - expected),
            duplicate_rows=int(frame.duplicated(["period_start", "period_end"], keep=False).sum()),
            invalid_values=int(bad_values.sum()), nonpositive_market=bool(name == "market" and frame.handle.le(0).any()),
            source_files=frame.get("source_file", pd.Series(dtype=str)).dropna().unique().tolist()))
    weekly_coverage = pd.DataFrame(checks)
    display(weekly_coverage)
    if any(c["missing_keys"] or c["unexpected_keys"] or c["duplicate_rows"] or c["invalid_values"] or c["nonpositive_market"] for c in checks):
        raise SystemExit("FAIL CLOSED: NY weekly coverage incomplete or conflicting; inspect weekly_coverage above.")
    fd, mk = fd.sort_values("period_start"), mk.sort_values("period_start")
    straddlers = fd_all[~((fd_all.period_start >= lo) & (fd_all.period_end <= hi))].sort_values(
        "period_start"
    )
    excl = "; ".join(
        f"{r.period_start.date()}→{r.period_end.date()}" for _, r in straddlers.iterrows()
    )
    first_start, last_end = fd.iloc[0].period_start, fd.iloc[-1].period_end
    return {
        "year": year,
        "complete_weeks": len(keys),
        "first_week": f"{first_start.date()}→{fd.iloc[0].period_end.date()}",
        "last_week": f"{keys[-1].date()}→{last_end.date()}",
        "covered_days": 7 * len(keys),
        "week_start_weekday": first_start.day_name(),
        "week_length_days": sorted(set((fd.period_end - fd.period_start).dt.days + 1)),
        "excluded_boundary_weeks": excl,
        "fd_handle": float(fd.handle.sum()),
        "mkt_handle": float(mk.handle.sum()),
    }


ny_meta = pd.DataFrame([ny_window(2025), ny_window(2026)])
if ny_meta.covered_days.nunique() != 1 or ny_meta.week_start_weekday.nunique() != 1:
    display(ny_meta)
    raise SystemExit("FAIL CLOSED: NY weekly comparison windows have different coverage.")
ny_meta["share"] = ny_meta["fd_handle"] / ny_meta["mkt_handle"]
ny_tbl = ny_meta[
    ["year", "complete_weeks", "first_week", "last_week", "covered_days", "fd_handle", "mkt_handle", "share"]
].copy()
ny_tbl["share_pp"] = 100 * (ny_tbl["share"] - ny_tbl["share"].shift(1))
display(Markdown("### NY weekly robustness (weeks fully inside Jan–Jul)"))
display(ny_tbl)
display(Markdown("### NY boundary / comparability check"))
display(
    ny_meta[
        [
            "year",
            "complete_weeks",
            "covered_days",
            "week_start_weekday",
            "week_length_days",
            "excluded_boundary_weeks",
        ]
    ]
)
print(
    f"NY complete-week share {100*ny_tbl.loc[0,'share']:.2f}% → {100*ny_tbl.loc[1,'share']:.2f}% "
    f"({ny_tbl.loc[1,'share_pp']:+.2f} pp). "
    f"Windows: {ny_meta.loc[0,'first_week'].split('→')[0]}–{ny_meta.loc[0,'last_week'].split('→')[1]} vs "
    f"{ny_meta.loc[1,'first_week'].split('→')[0]}–{ny_meta.loc[1,'last_week'].split('→')[1]}; "
    f"both {int(ny_meta.loc[0,'covered_days'])} covered days, Monday-start 7-day weeks; "
    "boundary weeks straddling Jan 1 / Jul 31 excluded in both years. "
    "Equal week counts alone are insufficient — duration and containment match. "
    "Weekly cash-basis; not pooled with monthly baseline."
)

# NJ revenue-only: retained rows do not establish a complete operator roster.
nj = raw[
    raw["state_code"].eq("NJ")
    & raw["channel"].eq(CHANNEL)
    & raw["frequency"].eq(FREQ)
    & raw["row_type"].eq("operator")
].copy()
nj_fd = nj[nj["operator"].isin(["Fanduel", "FANDUEL"])]


def nj_rev(months):
    ops = nj[nj["period_start"].isin(months)]
    fd = nj_fd[nj_fd["period_start"].isin(months)]
    checks = []
    for month in months:
        period, fan = ops[ops.period_start.eq(month)], fd[fd.period_start.eq(month)]
        revenue = period.gross_revenue
        valid_values = revenue.notna().all() and not revenue.isin([float("inf"), -float("inf")]).any()
        valid = len(fan) == 1 and not period.operator.duplicated().any() and valid_values and revenue.sum(min_count=1) > 0
        checks.append(dict(month=month, valid=bool(valid), fanduel_rows=len(fan), operator_rows=len(period),
            missing_values=int(revenue.isna().sum()), duplicate_rows=int(period.operator.duplicated(keep=False).sum()),
            fanduel_revenue=fan.gross_revenue.iloc[0] if len(fan) == 1 else float("nan"),
            observed_revenue=revenue.sum(min_count=1), native_operators=period.operator.tolist(),
            denominator_status="unverified observed operators",
            source_files=period.source_file.dropna().unique().tolist()))
    monthly_coverage = pd.DataFrame(checks)
    display(monthly_coverage)
    if monthly_coverage.empty or not monthly_coverage.valid.all():
        raise SystemExit("FAIL CLOSED: NJ monthly coverage incomplete or conflicting; inspect monthly_coverage above.")
    return float(fd["gross_revenue"].sum()), float(ops["gross_revenue"].sum())


a = nj_rev(CURRENT)
b = nj_rev(PRIOR)
if a and b:
    print(
        f"NJ exploratory ratio (unverified observed-operator denominator) Jan–Jul: "
        f"{100*b[0]/b[1]:.2f}% → {100*a[0]/a[1]:.2f}% ({100*(a[0]/a[1]-b[0]/b[1]):+.2f} pp). "
        "Not verified complete market share. Handle unavailable; label changed in 2026; not in handle baseline."
    )

# IN sensitivity
if window_ok(panels["IN"], CURRENT) and window_ok(panels["IN"], PRIOR):
    inn = state_summary(["IN"], CURRENT, PRIOR, "IN_derived")
    display(Markdown("### IN derived-denominator sensitivity (not in baseline)"))
    display(inn)


## 7. Baseline charts (retained set)


In [ ]:
# Heatmap Jan-Jul monthly share_pp
rows = []
for st in BASELINE:
    p = panels[st]
    for cur, pri in zip(CURRENT, PRIOR):
        a = p[p.month.eq(cur)].iloc[0]
        b = p[p.month.eq(pri)].iloc[0]
        rows.append({"state": st, "month": cur.strftime("%b"), "share_pp": 100 * (a.share - b.share)})
heat = pd.DataFrame(rows).pivot(index="state", columns="month", values="share_pp").reindex(BASELINE)[
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul"]
]
fig, ax = plt.subplots(figsize=(8.2, 2.6))
vmax = max(3.0, float(np.nanmax(np.abs(heat.to_numpy()))))
im = ax.imshow(heat.to_numpy(), cmap="coolwarm", aspect="auto", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(7), heat.columns)
ax.set_yticks(range(4), heat.index)
for i in range(4):
    for j in range(7):
        ax.text(j, i, f"{heat.iloc[i,j]:+.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Baseline exploratory: monthly FanDuel handle-share change (pp)")
fig.colorbar(im, ax=ax, fraction=0.05, pad=0.02)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 3.0))
ordr = base_jj.sort_values("share_pp")
ax.barh(ordr["state"], ordr["share_pp"], color=[C_ORANGE if v > 0 else C_BLUE for v in ordr.share_pp])
ax.axvline(0, color=C_BLACK, linewidth=0.9)
ax.set_title("Baseline Jan–Jul handle-share change")
ax.set_xlabel("pp")
fig.tight_layout()
plt.show()


## 8. Findings


In [ ]:
jm_b = timing[(timing["window"].eq("Jan-May")) & (timing["sample"].eq("baseline"))].iloc[0]
jj_b = timing[(timing["window"].eq("Jun-Jul")) & (timing["sample"].eq("baseline"))].iloc[0]
july_b = timing[(timing["window"].eq("July")) & (timing["sample"].eq("baseline"))].iloc[0]
jm_x = timing[(timing["window"].eq("Jan-May")) & (timing["sample"].eq("ex_KS"))].iloc[0]
jj_x = timing[(timing["window"].eq("Jun-Jul")) & (timing["sample"].eq("ex_KS"))].iloc[0]
july_x = timing[(timing["window"].eq("July")) & (timing["sample"].eq("ex_KS"))].iloc[0]
# improvement = less-negative YoY share change vs Jan–May; positive = share_pp > 0
imp_b = jj_b.pooled_share_pp > jm_b.pooled_share_pp
imp_x = jj_x.pooled_share_pp > jm_x.pooled_share_pp
pos_b = july_b.pooled_share_pp > 0
pos_x = july_x.pooled_share_pp > 0
m = month_pp.set_index("month")
t3_jul = t3_pool[t3_pool["t3m_end"].eq("2026-07")].iloc[0]
lines = [
    f"**Broad OSB weakness (baseline):** Jan–Jul share fell in {w_base['n_loss']}/{w_base['n']} states; pooled {w_base['pooled_share_pp']:+.2f} pp (ex-KS {w_ex['pooled_share_pp']:+.2f}). Mix effect negligible.",
    (
        f"**Monthly timing (not “no improvement before June” as a slogan):** "
        f"pooled YoY share_pp stayed deeply negative Jan–May "
        f"(Jan {m.loc['Jan','baseline_pp']:+.2f} … May {m.loc['May','baseline_pp']:+.2f} pp); "
        f"June improved to {m.loc['Jun','baseline_pp']:+.2f} pp; "
        f"only July turned positive on the baseline sample ({m.loc['Jul','baseline_pp']:+.2f} pp). "
        f"T3M ending Jul-2026 is still negative ({t3_jul.baseline_pp:+.2f} pp baseline; {t3_jul.ex_KS_pp:+.2f} ex-KS). "
        "World Cup dates overlap Jun–Jul 2026 — **not** a causal estimate."
    ),
    (
        f"**Improvement vs turning positive (ex-KS):** "
        f"Jan–May {jm_x.pooled_share_pp:+.2f} → Jun–Jul {jj_x.pooled_share_pp:+.2f} pp "
        f"({'improvement survives' if imp_x else 'improvement does not survive'} ex-KS). "
        f"July alone: baseline {july_b.pooled_share_pp:+.2f} pp "
        f"({'positive sign' if pos_b else 'not positive'}) vs ex-KS {july_x.pooled_share_pp:+.2f} pp "
        f"({'positive sign survives' if pos_x else 'positive sign does **not** survive'} ex-KS)."
    ),
    (f"**Expanded sample:** adding OH (transfer sum) → pooled Jan–Jul {w_oh['pooled_share_pp']:+.2f} pp; direction unchanged vs baseline." if w_oh is not None else "**Expanded sample:** OH matched coverage incomplete; no expanded result calculated."),
    (
        f"**NY weekly robustness:** {ny_tbl.loc[1,'share_pp']:+.2f} pp over matched fully-contained windows "
        f"({ny_meta.loc[0,'first_week']}…{ny_meta.loc[0,'last_week']} vs "
        f"{ny_meta.loc[1,'first_week']}…{ny_meta.loc[1,'last_week']}; "
        f"{int(ny_meta.loc[0,'covered_days'])} days each; boundary straddlers excluded)."
    ),
    "**Sports mix:** not identifiable in retained schema.",
    (
        "**Missing data by question:** "
        "(1) **recovery durability** — post-July 2026 months; "
        "(2) **denominator confidence** — printed IN statewide OSB totals; "
        "(3) **cross-product breadth** — brand-level FanDuel casino in NJ/PA with complete 2025–26 months; "
        "(4) **sports-mix attribution** — sport-level (ideally operator-by-sport) OSB handle."
    ),
]
display(Markdown("### Opening findings"))
for ln in lines:
    display(Markdown("- " + ln))
print(
    f"Diagnostic flags: improvement_exKS={imp_x} (also baseline={imp_b}); "
    f"july_positive_baseline={pos_b}; july_positive_exKS={pos_x}."
)


## 9. Integrity


In [ ]:
assert sha256(ORIGINAL_DB) == h0 and sha256(STAGING_DB) == s0
print("DB hashes unchanged", s0)
print("baseline", BASELINE)
print("charts rendered in this notebook section set")
